In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import json
pd.set_option('display.max_columns', None)


In [2]:
def get_mean_std(df, var_name):
    """
    Prints the mean and standard deviation (rounded to 2 decimal places) of the given variable name in the dataframe.
    Ignores NaN values for mean and std calculations.
    Also prints the percentage of NaN values in the column.
    """
    col = df[var_name]
    nan_percentage = col.isna().mean() * 100
    print(f"NaN percentage for {var_name}: {nan_percentage:.2f}%")
    mean_val = col.mean(skipna=True)
    std_val = col.std(skipna=True)
    print(f"Mean: {mean_val:.2f}, Std: {std_val:.2f}")


In [3]:
def get_value_counts_and_percent(df, var_name, value):
    """
    Prints the number and percentage of rows in df where var_name equals value.
    Also prints the percentage of NaN values in the column.
    """
    total = df.shape[0]
    count = (df[var_name] == value).sum()
    percent = (count / total) * 100 if total > 0 else 0
    nan_percentage = df[var_name].isna().mean() * 100
    print(f"Value '{value}' in '{var_name}': {count} ({percent:.2f}%)")
    print(f"NaN percentage for '{var_name}': {nan_percentage:.2f}%")


In [4]:
def print_value_counts(df, var_name):
    """
    Prints the count of each unique value in the specified variable (column) of the dataframe,
    sorted in increasing order of key.
    Also prints the percentage of NaN values in the column.
    """
    counts = df[var_name].value_counts(dropna=False).sort_index()
    nan_percentage = df[var_name].isna().mean() * 100
    print(f"Value counts for '{var_name}':")
    print(counts)
    print(f"NaN percentage for '{var_name}': {nan_percentage:.2f}%")


In [5]:
def anova_test(df, value_col, group_col):
    """
    Performs ANOVA test across groups defined by df[group_col] using the variable df[value_col].
    Only considers groups with at least one non-NaN value in value_col.
    Returns the p-value in scientific notation.
    """
    # Get unique, non-null groups from group_col
    uniq_groups = df[group_col].dropna().unique()
    samples = [df[df[group_col] == grp][value_col].dropna() for grp in uniq_groups]
    # Filter empty samples
    samples = [samp for samp in samples if len(samp) > 0]
    if len(samples) < 2:
        return 'N.A'
    test_stat, p_value = stats.f_oneway(*samples)
    return '{:.3e}'.format(p_value)

def chi_square_test(df, categorical_col, group_col):
    """
    Performs a chi-square test of independence for counts of categorical_col across groups in group_col.
    Returns the p-value in scientific notation.
    """
    # Get valid group labels, excluding NaNs
    uniq_groups = [g for g in df[group_col].dropna().unique()]
    if categorical_col not in df.columns or len(uniq_groups) < 2:
        return 'N.A'
    # Build contingency table
    # Only use non-NaN values for both group and categorical_col
    data = df[[categorical_col, group_col]].dropna()
    if data.empty:
        return 'N.A'
    contingency = pd.crosstab(data[group_col], data[categorical_col])
    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        return 'N.A'
    chi2, p, dof, exp = stats.chi2_contingency(contingency)
    return '{:.3e}'.format(p)
    # return p

# NACC

In [6]:
import warnings
warnings.filterwarnings("ignore")

nacc_train = pd.read_csv("../data/nacc/train.csv")
nacc_test = pd.read_csv("../data/nacc/test.csv")

In [7]:
nacc_merged = pd.concat([nacc_train, nacc_test], ignore_index=True)

In [8]:
print(len(nacc_merged))
print(len(nacc_merged['NACCID'].unique()))

48876
48876


In [9]:
with open("../data/nacc/NACC_dictionary.json", "r") as f:
    nacc_dict = json.load(f)

In [10]:
nacc_dict['NACCNIHR']['Values']["1"]

'White'

In [11]:
# nacc_merged = nacc_merged.replace([-4, -4.0, "-4", "-4.0"], np.nan)

In [12]:
nacc_merged['EDUC'] = nacc_merged['EDUC'].replace([99, "99"], np.nan)
nacc_merged["NACCNIHR"] = nacc_merged["NACCNIHR"].replace([99, "99"], np.nan)
nacc_merged["CDRGLOB"] = nacc_merged["CDRGLOB"].replace(99, np.nan)

In [13]:
def nacc_stats(df, group_label="Overall"):
    """
    Collects statistics for NACC dataset and returns a dictionary.
    Rounds values to 2 decimal places where appropriate.
    """
    stats_dict = {"Group": group_label}
    
    # Create a copy to avoid modifying the original dataframe
    df = df.copy()
    
    # AGE
    col = df["NACCAGE"]
    stats_dict["AGE_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["AGE_Mean"] = round(col.mean(skipna=True), 2) if not col.isna().all() else None
    stats_dict["AGE_Std"] = round(col.std(skipna=True), 2) if not col.isna().all() else None
    stats_dict["AGE_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["AGE_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # SEX
    col = df["SEX"]
    total = df.shape[0]
    count_female = (col == 2).sum()
    stats_dict["SEX_Female_Count"] = int(count_female)
    stats_dict["SEX_Female_Pct"] = round((count_female / total) * 100, 2) if total > 0 else 0
    stats_dict["SEX_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["SEX_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["SEX_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # EDUC
    # df['EDUC'] = df['EDUC'].replace([99, "99"], np.nan)
    col = df["EDUC"]
    stats_dict["EDUC_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["EDUC_Mean"] = round(col.mean(skipna=True), 2) if not col.isna().all() else None
    stats_dict["EDUC_Std"] = round(col.std(skipna=True), 2) if not col.isna().all() else None
    stats_dict["EDUC_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["EDUC_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # RACE
    # df["NACCNIHR"] = df["NACCNIHR"].replace([99, "99"], np.nan)
    col = df["NACCNIHR"]
    stats_dict["RACE_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["RACE_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["RACE_Max"] = round(col.max(), 2) if not col.isna().all() else None
    # Get value counts for RACE
    race_counts = col.value_counts(dropna=False).sort_index()
    for val, count in race_counts.items():
        if pd.isna(val):
            stats_dict["RACE_NaN_Count"] = int(count)
        else:
            stats_dict[f"RACE_{nacc_dict['NACCNIHR']['Values'][str(int(val))]}_Count"] = int(count)
    
    # CDR
    # df["CDRGLOB"] = df["CDRGLOB"].replace(99, np.nan)
    col = df["CDRGLOB"]
    stats_dict["CDR_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["CDR_Mean"] = round(col.mean(skipna=True), 2) if not col.isna().all() else None
    stats_dict["CDR_Std"] = round(col.std(skipna=True), 2) if not col.isna().all() else None
    stats_dict["CDR_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["CDR_Max"] = round(col.max(), 2) if not col.isna().all() else None

    return stats_dict

In [14]:
# Collect statistics for overall dataset
nacc_stats_list = []
nacc_stats_list.append(nacc_stats(nacc_merged, group_label="Overall"))

In [15]:
# Collect statistics for each NACCUDSD subgroup
for value in sorted(nacc_merged['NACCUDSD'].dropna().unique()):
    subgroup_df = nacc_merged[nacc_merged['NACCUDSD'] == value].reset_index(drop=True)
    nacc_stats_list.append(nacc_stats(subgroup_df, group_label=f"{nacc_dict['NACCUDSD']['Values'][str(value)]}"))

In [16]:
nacc_merged['NACCUDSD'].value_counts()

NACCUDSD
4    21094
1    18459
3     9323
Name: count, dtype: int64

In [17]:
# Create DataFrame from all collected statistics
nacc_stats_df = pd.DataFrame(nacc_stats_list)
nacc_stats_df

,Group,AGE_NaN_pct,AGE_Mean,AGE_Std,AGE_Min,AGE_Max,SEX_Female_Count,SEX_Female_Pct,SEX_NaN_pct,SEX_Min,SEX_Max,EDUC_NaN_pct,EDUC_Mean,EDUC_Std,EDUC_Min,EDUC_Max,RACE_NaN_pct,RACE_Min,RACE_Max,RACE_White_Count,RACE_Black or African American_Count,RACE_American Indian or Alaska Native_Count,RACE_Native Hawaiian or Pacific Islander_Count,RACE_Asian_Count,RACE_Multiracial_Count,RACE_NaN_Count,CDR_NaN_pct,CDR_Mean,CDR_Std,CDR_Min,CDR_Max
0,Overall,0.0,74.53,11.02,18,110,27940,57.17,0.0,1,2,0.76,15.24,3.40,0.0,31.0,1.77,1.0,6.0,38482,6344,306,35,1332,1513,864,0.01,0.80,0.94,0.0,3.0
1,Normal cognition,0.0,72.60,11.45,18,108,12012,65.07,0.0,1,2,0.48,15.88,2.97,0.0,29.0,1.43,1.0,6.0,13981,2835,130,12,596,641,264,0.03,0.05,0.16,0.0,3.0
2,MCI,0.0,75.45,10.12,21,106,4959,53.19,0.0,1,2,0.74,15.20,3.41,0.0,31.0,1.74,1.0,6.0,6893,1558,71,4,294,341,162,0.01,0.46,0.19,0.0,3.0
3,Dementia,0.0,75.82,10.77,25,110,10969,52.00,0.0,1,2,1.02,14.71,3.66,0.0,30.0,2.08,1.0,6.0,17608,1951,105,19,442,531,438,0.00,1.60,0.91,0.0,3.0


In [18]:
nacc_stats_df.to_csv("../data/stats/nacc.csv", index=False)

In [19]:
# Collect statistical test results
nacc_test_results = {
    "Group": "NACC",
    "AGE_ANOVA_pvalue": anova_test(nacc_merged, "NACCAGE", "NACCUDSD"),
    "SEX_ChiSquare_pvalue": chi_square_test(nacc_merged, "SEX", "NACCUDSD"),
    "EDUC_ANOVA_pvalue": anova_test(nacc_merged, "EDUC", "NACCUDSD"),
    "RACE_ChiSquare_pvalue": chi_square_test(nacc_merged, "NACCNIHR", "NACCUDSD"),
    "CDR_ChiSquare_pvalue": chi_square_test(nacc_merged, "CDRGLOB", "NACCUDSD")
}
# nacc_stats_list.append(test_results)
nacc_test_results


{'Group': 'NACC',
 'AGE_ANOVA_pvalue': '3.467e-202',
 'SEX_ChiSquare_pvalue': '4.554e-166',
 'EDUC_ANOVA_pvalue': '8.971e-254',
 'RACE_ChiSquare_pvalue': '2.631e-127',
 'CDR_ChiSquare_pvalue': '0.000e+00'}

# NIFD

In [20]:
nifd = pd.read_csv("../data/nifd/test_gamlss.csv")

In [21]:
len(nifd)

292

In [22]:
with open("../data/nifd/NIFD_dictionary.json", "r") as f:
    nifd_dict = json.load(f)

In [23]:
nifd_dict['RACE']['Values']

{'1': 'White',
 '2': 'Black/African-American',
 '3': 'Asian',
 '4': 'Pacific Islander',
 '5': 'Multi-racial',
 '6': 'Not declared'}

In [24]:
def add_cogstate(row):
    if row['DX'] in ['BV', 'PNFA', 'SV']:
        row['NACCUDSD'] = 4
    elif row['DX'] == 'CON':
        row['NACCUDSD'] = 1
        
    return row
        
nifd = nifd.apply(add_cogstate, axis=1)

In [25]:
nifd_dict['NACCUDSD'] = {
    "FORM ID": "D1",
    "Description": "Cognitive status at UDS visit",
    "Values": {
        "1": "Normal cognition",
        "2": "Impaired-not-MCI",
        "3": "MCI",
        "4": "Dementia"
    }
}

In [26]:
nifd['NACCUDSD'].value_counts()

NACCUDSD
4    156
1    136
Name: count, dtype: int64

In [28]:
nifd['EDUCATION'] = nifd['EDUCATION'].replace([99, "99"], np.nan)
nifd["RACE"] = nifd["RACE"].replace([50, 99], np.nan)
nifd["CDR_TOT"] = nifd["CDR_TOT"].replace(-5.0, np.nan)

In [29]:
def nifd_stats(df, group_label="Overall"):
    """
    Collects statistics for NACC dataset and returns a dictionary.
    Rounds values to 2 decimal places where appropriate.
    """
    stats_dict = {"Group": group_label}
    
    # Create a copy to avoid modifying the original dataframe
    df = df.copy()
    
    # AGE
    col = df["AGE"]
    stats_dict["AGE_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["AGE_Mean"] = round(col.mean(skipna=True), 2) if not col.isna().all() else None
    stats_dict["AGE_Std"] = round(col.std(skipna=True), 2) if not col.isna().all() else None
    stats_dict["AGE_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["AGE_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # SEX
    col = df["GENDER"]
    total = df.shape[0]
    count_female = (col == 2).sum()
    stats_dict["SEX_Female_Count"] = int(count_female)
    stats_dict["SEX_Female_Pct"] = round((count_female / total) * 100, 2) if total > 0 else 0
    stats_dict["SEX_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["SEX_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["SEX_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # EDUC
    # df['EDUCATION'] = df['EDUCATION'].replace([99, "99"], np.nan)
    col = df["EDUCATION"]
    stats_dict["EDUC_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["EDUC_Mean"] = round(col.mean(skipna=True), 2) if not col.isna().all() else None
    stats_dict["EDUC_Std"] = round(col.std(skipna=True), 2) if not col.isna().all() else None
    stats_dict["EDUC_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["EDUC_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # RACE
    # df["RACE"] = df["RACE"].replace([50, 99], np.nan)
    col = df["RACE"]
    stats_dict["RACE_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["RACE_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["RACE_Max"] = round(col.max(), 2) if not col.isna().all() else None
    # Get value counts for RACE
    race_counts = col.value_counts(dropna=False).sort_index()
    for val, count in race_counts.items():
        if pd.isna(val):
            stats_dict["RACE_NaN_Count"] = int(count)
        else:
            stats_dict[f"RACE_{nifd_dict['RACE']['Values'][str(int(val))]}_Count"] = int(count)
    
    # CDR
    # df["CDR_TOT"] = df["CDR_TOT"].replace(-5.0, np.nan)
    col = df["CDR_TOT"]
    stats_dict["CDR_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["CDR_Mean"] = round(col.mean(skipna=True), 2) if not col.isna().all() else None
    stats_dict["CDR_Std"] = round(col.std(skipna=True), 2) if not col.isna().all() else None
    stats_dict["CDR_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["CDR_Max"] = round(col.max(), 2) if not col.isna().all() else None

    return stats_dict

In [30]:
# Collect statistics for overall dataset
nifd_stats_list = []
nifd_stats_list.append(nifd_stats(nifd, group_label="Overall"))

In [31]:
# Collect statistics for each NACCUDSD subgroup
for value in sorted(nifd['NACCUDSD'].dropna().unique()):
    subgroup_df = nifd[nifd['NACCUDSD'] == value].reset_index(drop=True)
    nifd_stats_list.append(nifd_stats(subgroup_df, group_label=f"{nifd_dict['NACCUDSD']['Values'][str(value)]}"))

In [32]:
len(nifd)

292

In [33]:
nifd['NACCUDSD'].value_counts()

NACCUDSD
4    156
1    136
Name: count, dtype: int64

In [34]:
nifd['CDR_TOT'].replace(-5.0, np.nan).isna().mean()

0.2671232876712329

In [35]:
# Create DataFrame from all collected statistics
nifd_stats_list = pd.DataFrame(nifd_stats_list)
nifd_stats_list

,Group,AGE_NaN_pct,AGE_Mean,AGE_Std,AGE_Min,AGE_Max,SEX_Female_Count,SEX_Female_Pct,SEX_NaN_pct,SEX_Min,SEX_Max,EDUC_NaN_pct,EDUC_Mean,EDUC_Std,EDUC_Min,EDUC_Max,RACE_NaN_pct,RACE_Min,RACE_Max,RACE_White_Count,RACE_Black/African-American_Count,RACE_Multi-racial_Count,RACE_NaN_Count,CDR_NaN_pct,CDR_Mean,CDR_Std,CDR_Min,CDR_Max
0,Overall,0.0,65.85,7.82,39.05,87.98,142,48.63,0.0,1,2,3.77,16.52,2.53,8.0,20.0,8.90,1.0,5.0,253,1.0,12,26,26.71,0.87,0.90,0.0,3.0
1,Normal cognition,0.0,66.40,8.64,39.05,87.98,78,57.35,0.0,1,2,5.15,17.32,1.92,12.0,20.0,16.91,1.0,5.0,109,NaN,4,23,47.06,0.01,0.08,0.0,0.5
2,Dementia,0.0,65.37,7.03,45.90,83.25,64,41.03,0.0,1,2,2.56,15.84,2.79,8.0,20.0,1.92,1.0,5.0,144,1.0,8,3,8.97,1.31,0.81,0.0,3.0


In [36]:
nifd_stats_list.to_csv("../data/stats/nifd.csv", index=False)

In [37]:
# Collect statistical test results
nifd_test_results = {
    "Group": "NIFD",
    "AGE_ANOVA_pvalue": anova_test(nifd, "AGE", "NACCUDSD"),
    "SEX_ChiSquare_pvalue": chi_square_test(nifd, "GENDER", "NACCUDSD"),
    "EDUC_ANOVA_pvalue": anova_test(nifd, "EDUCATION", "NACCUDSD"),
    "RACE_ChiSquare_pvalue": chi_square_test(nifd, "RACE", "NACCUDSD"),
    "CDR_ChiSquare_pvalue": chi_square_test(nifd, "CDR_TOT", "NACCUDSD")
}
# nacc_stats_list.append(test_results)
nifd_test_results


{'Group': 'NIFD',
 'AGE_ANOVA_pvalue': '2.634e-01',
 'SEX_ChiSquare_pvalue': '7.650e-03',
 'EDUC_ANOVA_pvalue': '6.880e-07',
 'RACE_ChiSquare_pvalue': '5.524e-01',
 'CDR_ChiSquare_pvalue': '5.211e-38'}

# PPMI

In [38]:
ppmi = pd.read_csv("../data/ppmi/test_gamlss.csv")

In [39]:
len(ppmi)

3315

In [40]:
with open("../data/ppmi/PPMI_dictionary.json", "r") as f:
    ppmi_dict = json.load(f)

In [41]:
ppmi_dict['race']['Values']

{'1': 'White',
 '2': 'Black',
 '3': 'Asian',
 '4': 'Other (includes multi-racial)',
 '.': 'Unknown/Not reported'}

In [42]:
ppmi['cogstate'].value_counts()

cogstate
1.0    2757
2.0     495
3.0      63
Name: count, dtype: int64

In [43]:
def ppmi_stats(df, group_label="Overall"):
    """
    Collects statistics for NACC dataset and returns a dictionary.
    Rounds values to 2 decimal places where appropriate.
    """
    stats_dict = {"Group": group_label}
    
    # Create a copy to avoid modifying the original dataframe
    df = df.copy()
    
    # AGE
    col = df["age"]
    stats_dict["AGE_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["AGE_Mean"] = round(col.mean(skipna=True), 2) if not col.isna().all() else None
    stats_dict["AGE_Std"] = round(col.std(skipna=True), 2) if not col.isna().all() else None
    stats_dict["AGE_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["AGE_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # SEX
    col = df["SEX"]
    total = df.shape[0]
    count_female = (col == 0).sum()
    stats_dict["SEX_Female_Count"] = int(count_female)
    stats_dict["SEX_Female_Pct"] = round((count_female / total) * 100, 2) if total > 0 else 0
    stats_dict["SEX_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["SEX_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["SEX_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # EDUC
    col = df["EDUCYRS"]
    stats_dict["EDUC_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["EDUC_Mean"] = round(col.mean(skipna=True), 2) if not col.isna().all() else None
    stats_dict["EDUC_Std"] = round(col.std(skipna=True), 2) if not col.isna().all() else None
    stats_dict["EDUC_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["EDUC_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # RACE
    col = df["race"]
    stats_dict["RACE_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["RACE_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["RACE_Max"] = round(col.max(), 2) if not col.isna().all() else None
    # Get value counts for RACE
    race_counts = col.value_counts(dropna=False).sort_index()
    for val, count in race_counts.items():
        if pd.isna(val):
            stats_dict["RACE_NaN_Count"] = int(count)
        else:
            stats_dict[f"RACE_{ppmi_dict['race']['Values'][str(int(val))]}_Count"] = int(count)
    
    # CDR
    stats_dict["CDR_NaN_pct"] = np.NaN
    stats_dict["CDR_Mean"] = np.NaN
    stats_dict["CDR_Std"] = np.NaN
    stats_dict["CDR_Min"] = np.NaN
    stats_dict["CDR_Max"] = np.NaN

    return stats_dict

In [44]:
# Collect statistics for overall dataset
ppmi_stats_list = []
ppmi_stats_list.append(ppmi_stats(ppmi, group_label="Overall"))

In [45]:
# Collect statistics for each NACCUDSD subgroup
for value in sorted(ppmi['cogstate'].dropna().unique()):
    subgroup_df = ppmi[ppmi['cogstate'] == value].reset_index(drop=True)
    ppmi_stats_list.append(ppmi_stats(subgroup_df, group_label=f"{ppmi_dict['Diagnosis']['cogstate']['Values'][str(int(value))]}"))

In [46]:
ppmi[ppmi['cogstate'].isna()]

,SITE,PATNO,COHORT,subgroup,enroll_phase,HIQ_RBD,study_status,NSD_Status,NSD_STAGE,PRIMDIAG,OTHNEURO,EVENT_ID,YEAR,visit_date,age,age_at_visit,SEX,EDUCYRS,race,HISPLAT,ASHKJEW,AFICBERB,BASQUE,fampd,fampd_bin,handed,howlive,sex_orient,BMI,agediag,ageonset,duration,duration_yrs,DOMSIDE,sym_tremor,sym_rigid,sym_brady,sym_posins,sym_other,sym_unknown,PDTRTMNT,LEDD,age_DATSCAN,age_LP,age_upsit,upsit,upsit_pctl,upsit_pctl15,moca,bjlot,clockdraw,hvlt_discrimination,hvlt_immediaterecall,hvlt_retention,HVLTFPRL,HVLTRDLY,HVLTREC,lexical,lns,MODBNT,SDMTOTAL,TMT_A,TMT_B,VLTANIM,MCI_testscores,cogstate,MSEADLG,quip,quip_any,quip_gamble,quip_sex,quip_buy,quip_eat,quip_hobby,quip_pund,quip_walk,ess,rem,gds,stai,stai_state,stai_trait,scopa,scopa_gi,scopa_ur,scopa_cv,scopa_therm,scopa_pm,scopa_sex,orthostasis,hy,hy_on,NHY,NHY_ON,pigd,pigd_on,td_pigd,td_pigd_on,NP1COG,NP1HALL,NP1DPRS,NP1ANXS,NP1APAT,NP1DDS,NP1FATG,updrs1_score,updrs2_score,updrs3_score,updrs3_score_on,updrs4_score,updrs_totscore,updrs_totscore_on,pm_any,pm_adl_any,pm_fd_any,pm_auto_any,pm_cog_any,pm_mc_any,pm_wb_any,CSFSAA,CSFSAA_assay,abeta,abeta_LLOD,abeta_ULOD,tau,tau_LLOD,ptau,ptau_LLOD,asyn,hemohi,urate,total_di_18_1_BMP,total_di_22_6_BMP,_2_2__di_22_6_BMP,NFL_CSF,nfl_serum,APOE,APOE_e4,lowput_expected,DATSCAN_CAUDATE_L,DATSCAN_CAUDATE_R,con_caudate,ips_caudate,mean_caudate,DATSCAN_PUTAMEN_L,DATSCAN_PUTAMEN_R,con_putamen,ips_putamen,mean_putamen,con_striatum,ips_striatum,mean_striatum,Stage_partial_UPDRS1,Stage_subpark,Stage_PDTreat,Stage_S,Stage_D,Stage_G,patient_summary,diag_summary,brain_findings_summary


In [47]:
# Create DataFrame from all collected statistics
ppmi_stats_list = pd.DataFrame(ppmi_stats_list)
ppmi_stats_list

,Group,AGE_NaN_pct,AGE_Mean,AGE_Std,AGE_Min,AGE_Max,SEX_Female_Count,SEX_Female_Pct,SEX_NaN_pct,SEX_Min,SEX_Max,EDUC_NaN_pct,EDUC_Mean,EDUC_Std,EDUC_Min,EDUC_Max,RACE_NaN_pct,RACE_Min,RACE_Max,RACE_White_Count,RACE_Black_Count,RACE_Asian_Count,RACE_Other (includes multi-racial)_Count,RACE_NaN_Count,CDR_NaN_pct,CDR_Mean,CDR_Std,CDR_Min,CDR_Max
0,Overall,0.0,64.57,8.78,26.44,87.39,1483,44.74,0.0,0,1,0.84,16.11,3.00,0.0,20.0,0.66,1.0,4.0,3096,53,32,112,22.0,NaN,NaN,NaN,NaN,NaN
1,Normal Cognition,0.0,64.01,8.75,26.44,87.39,1294,46.94,0.0,0,1,1.02,16.21,2.90,0.0,20.0,0.69,1.0,4.0,2574,44,28,92,19.0,NaN,NaN,NaN,NaN,NaN
2,Mild Cognitive Impairment,0.0,67.30,8.28,44.42,87.37,170,34.34,0.0,0,1,0.00,15.58,3.33,3.0,20.0,0.61,1.0,4.0,464,8,3,17,3.0,NaN,NaN,NaN,NaN,NaN
3,Dementia,0.0,67.76,9.38,32.21,84.72,19,30.16,0.0,0,1,0.00,15.83,4.12,5.0,20.0,0.00,1.0,4.0,58,1,1,3,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
ppmi_stats_list.to_csv("../data/stats/ppmi.csv", index=False)

In [49]:
len(ppmi)

3315

In [50]:
# Collect statistical test results
ppmi_test_results = {
    "Group": "PPMI",
    "AGE_ANOVA_pvalue": anova_test(ppmi, "age", "cogstate"),
    "SEX_ChiSquare_pvalue": chi_square_test(ppmi, "SEX", "cogstate"),
    "EDUC_ANOVA_pvalue": anova_test(ppmi, "EDUCYRS", "cogstate"),
    "RACE_ChiSquare_pvalue": chi_square_test(ppmi, "race", "cogstate"),
    # "CDR_ChiSquare_pvalue": chi_square_test(nifd, "CDR_TOT", "NACCUDSD")
}
# nacc_stats_list.append(test_results)
ppmi_test_results


{'Group': 'PPMI',
 'AGE_ANOVA_pvalue': '1.615e-15',
 'SEX_ChiSquare_pvalue': '9.065e-08',
 'EDUC_ANOVA_pvalue': '7.121e-05',
 'RACE_ChiSquare_pvalue': '9.674e-01'}

# ADNI

In [51]:
adni = pd.read_csv("../data/adni/test_gamlss.csv")

In [52]:
len(adni)

1392

In [53]:
with open("../data/adni/adni_dictionary_oct2025_vj.json", "r") as f:
    adni_dict = json.load(f)

In [54]:
adni_dict['PTRACCAT']['Values']

{'1': 'American Indian or Alaskan Native',
 '2': 'Asian',
 '3': 'Native Hawaiian or Other Pacific Islander',
 '4': 'Black or African American',
 '5': 'White',
 '6': 'More than one race',
 '7': 'Unknown'}

In [55]:
adni['PTRACCAT'].value_counts()

PTRACCAT
5.0    1235
4.0      95
2.0      29
6.0      20
7.0       8
1.0       3
3.0       2
Name: count, dtype: int64

In [56]:
adni['PTRACCAT'] = adni['PTRACCAT'].replace([7, "7"], np.nan)

In [57]:
def adni_stats(df, group_label="Overall"):
    """
    Collects statistics for NACC dataset and returns a dictionary.
    Rounds values to 2 decimal places where appropriate.
    """
    stats_dict = {"Group": group_label}
    
    # Create a copy to avoid modifying the original dataframe
    df = df.copy()
    
    # AGE
    col = df["AGE_MRI"]
    stats_dict["AGE_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["AGE_Mean"] = round(col.mean(skipna=True), 2) if not col.isna().all() else None
    stats_dict["AGE_Std"] = round(col.std(skipna=True), 2) if not col.isna().all() else None
    stats_dict["AGE_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["AGE_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # SEX
    col = df["PTGENDER"]
    total = df.shape[0]
    count_female = (col == 2).sum()
    stats_dict["SEX_Female_Count"] = int(count_female)
    stats_dict["SEX_Female_Pct"] = round((count_female / total) * 100, 2) if total > 0 else 0
    stats_dict["SEX_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["SEX_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["SEX_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # EDUC
    col = df["PTEDUCAT"]
    stats_dict["EDUC_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["EDUC_Mean"] = round(col.mean(skipna=True), 2) if not col.isna().all() else None
    stats_dict["EDUC_Std"] = round(col.std(skipna=True), 2) if not col.isna().all() else None
    stats_dict["EDUC_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["EDUC_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # RACE
    col = df["PTRACCAT"]
    stats_dict["RACE_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["RACE_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["RACE_Max"] = round(col.max(), 2) if not col.isna().all() else None
    # Get value counts for RACE
    race_counts = col.value_counts(dropna=False).sort_index()
    for val, count in race_counts.items():
        if pd.isna(val):
            stats_dict["RACE_NaN_Count"] = int(count)
        else:
            stats_dict[f"RACE_{adni_dict['PTRACCAT']['Values'][str(int(val))]}_Count"] = int(count)
    
    # CDR
    col = df['CDMEMORY']
    stats_dict["CDR_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["CDR_Mean"] = round(col.mean(skipna=True), 2) if not col.isna().all() else None
    stats_dict["CDR_Std"] = round(col.std(skipna=True), 2) if not col.isna().all() else None
    stats_dict["CDR_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["CDR_Max"] = round(col.max(), 2) if not col.isna().all() else None

    return stats_dict

In [58]:
# Collect statistics for overall dataset
adni_stats_list = []
adni_stats_list.append(adni_stats(adni, group_label="Overall"))

In [59]:
# Collect statistics for each NACCUDSD subgroup
for value in sorted(adni['DIAGNOSIS'].dropna().unique()):
    subgroup_df = adni[adni['DIAGNOSIS'] == value].reset_index(drop=True)
    adni_stats_list.append(adni_stats(subgroup_df, group_label=f"{adni_dict['DIAGNOSIS']['Values'][str(int(value))]}"))

In [60]:
adni[adni['DIAGNOSIS'].isna()]

,RID,DIAGNOSIS,PTGENDER,PTDOBMM,PTDOBYY,PTEDUCAT,PTTLANG,PTETHCAT,PTRACCAT,CENTILOIDS,MMDATE,MMYEAR,MMMONTH,MMDAY,MMSEASON,MMHOSPIT,MMFLOOR,MMCITY,MMAREA,MMSTATE,MMWATCH,MMPENCIL,MMREPEAT,MMHAND,MMFOLD,MMONFLR,MMREAD,MMWRITE,MMDRAW,MMSCORE,WORD1,WORD1DL,WORD2,WORD2DL,WORD3,WORD3DL,WORDLIST,WORLDSCORE,CDMEMORY,CDORIENT,CDJUDGE,CDCOMMUN,CDHOME,CDCARE,CDGLOBAL,CDSOB,VSWEIGHT,VSWTUNIT,VSHEIGHT,VSHTUNIT,VSBPSYS,VSBPDIA,VSPULSE,VSRESP,VSTEMP,VSTMPSRC,VSTMPUNT,TRAILS,CUBE,CLOCKCON,CLOCKNO,CLOCKHAN,LION,RHINO,CAMEL,IMMT1W1,IMMT1W2,IMMT1W3,IMMT1W4,IMMT1W5,IMMT2W1,IMMT2W2,IMMT2W3,IMMT2W4,IMMT2W5,DIGFOR,DIGBACK,LETTERS,SERIAL1,SERIAL2,SERIAL3,SERIAL4,SERIAL5,REPEAT1,REPEAT2,FFLUENCY,ABSTRAN,ABSMEAS,DELW1,DELW2,DELW3,DELW4,DELW5,MONTH,YEAR,DAY,PLACE,CITY,MOCA,GDSOURCE,GDUNABL,GDSATIS,GDDROP,GDEMPTY,GDBORED,GDSPIRIT,GDAFRAID,GDHAPPY,GDHELP,GDHOME,GDMEMORY,GDALIVE,GDWORTH,GDENERGY,GDHOPE,GDBETTER,GDTOTAL,NPISOURCE,NPIA,NPIASEV,NPIB,NPIBSEV,NPIC,NPICSEV,NPID,NPIDSEV,NPIE,NPIESEV,NPIF,NPIFSEV,NPIG,NPIGSEV,NPIH,NPIHSEV,NPII,NPIISEV,NPIJ,NPIJSEV,NPIK,NPIKSEV,NPIL,NPILSEV,NPISCORE,FAQSOURCE,FAQFINAN,FAQFORM,FAQSHOP,FAQGAME,FAQBEVG,FAQMEAL,FAQEVENT,FAQTV,FAQREM,FAQTRAVL,FAQTOTAL,MOTHALIVE,MOTHAGE,MOTHDEM,MOTHAD,MOTHSXAGE,FATHALIVE,FATHAGE,FATHDEM,FATHAD,FATHSXAGE,baseline_health_hx,PLASMAPTAU181,Abeta_4240,HMONSET,HMSTEPWS,HMSOMATC,HMEMOTIO,HMHYPERT,HMSTROKE,HMNEURSM,HMNEURSG,HMSCORE,DATE,CLOCKCIRC,CLOCKSYM,CLOCKNUM,CLOCKHAND,CLOCKTIME,CLOCKSCOR,COPYCIRC,COPYSYM,COPYNUM,COPYHAND,COPYTIME,COPYSCOR,LMSTORY,LIMMTOTAL,AVTOT1,AVERR1,AVTOT2,AVERR2,AVTOT3,AVERR3,AVTOT4,AVERR4,AVTOT5,AVERR5,AVTOT6,AVERR6,AVTOTB,AVERRB,CATANIMSC,CATANPERS,CATANINTR,TRAASCOR,TRAAERRCOM,TRAAERROM,TRABSCOR,TRABERRCOM,TRABERROM,LDELTOTAL,LDELCUE,BNTSPONT,BNTSTIM,BNTCSTIM,BNTPHON,BNTCPHON,BNTTOTAL,AVDEL30MIN,AVDELERR1,AVDELTOT,AVDELERR2,ANARTND,ANARTERR,MINTSEMCUE,MINTTOTAL,MINTUNCUED,RAVLT_forgetting,RAVLT_immediate,RAVLT_learning,RAVLT_perc_forgetting,APGEN1,APGEN2,PHC_AB42,PHC_pTau,AT_class,AGE_MRI,AMYLOID_PET_POSITIVE,brain_findings_summary,patient_summary,diag_summary


In [61]:
adni['DIAGNOSIS'].value_counts()

DIAGNOSIS
2.0    603
1.0    555
3.0    234
Name: count, dtype: int64

In [62]:
# Create DataFrame from all collected statistics
adni_stats_list = pd.DataFrame(adni_stats_list)
adni_stats_list

,Group,AGE_NaN_pct,AGE_Mean,AGE_Std,AGE_Min,AGE_Max,SEX_Female_Count,SEX_Female_Pct,SEX_NaN_pct,SEX_Min,SEX_Max,EDUC_NaN_pct,EDUC_Mean,EDUC_Std,EDUC_Min,EDUC_Max,RACE_NaN_pct,RACE_Min,RACE_Max,RACE_American Indian or Alaskan Native_Count,RACE_Asian_Count,RACE_Native Hawaiian or Other Pacific Islander_Count,RACE_Black or African American_Count,RACE_White_Count,RACE_More than one race_Count,RACE_NaN_Count,CDR_NaN_pct,CDR_Mean,CDR_Std,CDR_Min,CDR_Max
0,Overall,0.0,73.45,7.64,55.04,95.98,701,50.36,0.0,1.0,2.0,0.0,16.31,2.57,6.0,20.0,0.57,1.0,6.0,3.0,29,2.0,95,1235,20,8.0,0.07,0.42,0.46,-1.0,3.0
1,Cognitively normal,0.0,73.03,7.10,55.16,95.44,321,57.84,0.0,1.0,2.0,0.0,16.72,2.42,6.0,20.0,0.54,1.0,6.0,2.0,17,NaN,58,464,11,3.0,0.00,0.01,0.08,0.0,0.5
2,MCI,0.0,73.05,7.70,55.04,92.15,278,46.10,0.0,1.0,2.0,0.0,16.19,2.63,8.0,20.0,0.83,1.0,6.0,1.0,4,2.0,29,556,6,5.0,0.00,0.53,0.22,-1.0,2.0
3,Dementia,0.0,75.46,8.41,55.26,95.98,102,43.59,0.0,1.0,2.0,0.0,15.64,2.61,9.0,20.0,0.00,2.0,6.0,NaN,8,NaN,8,215,3,NaN,0.43,1.11,0.46,0.5,3.0


In [63]:
adni_stats_list.to_csv("../data/stats/adni.csv", index=False)

In [64]:
# Collect statistical test results
adni_test_results = {
    "Group": "ADNI",
    "AGE_ANOVA_pvalue": anova_test(adni, "AGE_MRI", "DIAGNOSIS"),
    "SEX_ChiSquare_pvalue": chi_square_test(adni, "PTGENDER", "DIAGNOSIS"),
    "EDUC_ANOVA_pvalue": anova_test(adni, "PTEDUCAT", "DIAGNOSIS"),
    "RACE_ChiSquare_pvalue": chi_square_test(adni, "PTRACCAT", "DIAGNOSIS"),
    "CDR_ChiSquare_pvalue": chi_square_test(adni, "CDMEMORY", "DIAGNOSIS")
}
# nacc_stats_list.append(test_results)
adni_test_results


{'Group': 'ADNI',
 'AGE_ANOVA_pvalue': '5.369e-05',
 'SEX_ChiSquare_pvalue': '2.650e-05',
 'EDUC_ANOVA_pvalue': '1.487e-07',
 'RACE_ChiSquare_pvalue': '5.459e-05',
 'CDR_ChiSquare_pvalue': '0.000e+00'}

# Brainlat

In [65]:
brainlat_orig = pd.read_csv("/projectnb/vkolagrp/ketanss/adrdfm-2/brainlat/MRI/MRIdata_oct_7.csv")
brainlat = pd.read_csv("../data/brainlat/test_gamlss.csv")
brainlat_orig = brainlat_orig[brainlat_orig['MRI_ID'].isin(brainlat["ID"])].reset_index(drop=True)

In [66]:
brainlat_merged = pd.merge(brainlat, brainlat_orig, left_on="ID", right_on="MRI_ID", how="inner")


In [67]:
brainlat_merged.head()

,Diagnosis,Metadata,EEG Report,MRI Report,patient_summary,ID,MRI_ID,EEG_ID,diagnosis,sex,Age,years_education,laterality,moca_total,moca_visuospatial,moca_recog,moca_attention,moca_language,moca_abstraction,moca_memory,moca_orientation,ifs_total_score,ifs_motor_series,ifs_conflicting_instructions,ifs_motor_inhibition,ifs_digits,ifs_months,ifs_visual_wm,ifs_proverb,ifs_verbal_inhibition,mini_sea_fer,mini_sea_tom,emotion recog,T1,Rest,DWI,MF,eeg,paths
0,Healthy,"{""Sex"": ""Male"", ""Age"": 76.0, ""Years of educati...","{""EEG features extraction procedure"": ""How EEG...",NaN,"{""Metadata"": {""Sex"": ""Male"", ""Age"": 76.0, ""Yea...",sub-CLB10013,sub-CLB10013,sub-100039,CN,1,76.0,3.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,1.0,['/projectnb/vkolagrp/datasets/BrainLat/data/M...
1,Alzheimer's disease,"{""Sex"": ""Female"", ""Age"": 79.0, ""Years of educa...",NaN,"{""MRI features extraction procedure"": ""A Gener...","{""Metadata"": {""Sex"": ""Female"", ""Age"": 79.0, ""Y...",sub-CLB00123,sub-CLB00123,NaN,AD,0,79.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,1.0,3.0,3.0,3.0,1.0,2.0,1.0,6.0,6.9,10.9,NaN,1.0,1.0,0.0,1.5,0.0,['/projectnb/vkolagrp/datasets/BrainLat/data/M...
2,Healthy,"{""Sex"": ""Male"", ""Age"": 59.0, ""Years of educati...",NaN,NaN,"{""Metadata"": {""Sex"": ""Male"", ""Age"": 59.0, ""Yea...",sub-AR00024,sub-AR00024,NaN,CN,1,59.0,20.0,1.0,26.0,5.0,3.0,6.0,3.0,2.0,1.0,6.0,28.0,3.0,3.0,3.0,6.0,2.0,4.0,3.0,4.0,14.1,12.8,NaN,1.0,1.0,1.0,3.0,0.0,NaN
3,Healthy,"{""Sex"": ""Female"", ""Age"": 62.0, ""Years of educa...",NaN,"{""MRI features extraction procedure"": ""A Gener...","{""Metadata"": {""Sex"": ""Female"", ""Age"": 62.0, ""Y...",sub-CLB00103,sub-CLB00103,NaN,CN,0,62.0,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.0,3.0,3.0,3.0,4.0,2.0,3.0,3.0,6.0,12.4,14.6,NaN,1.0,0.0,1.0,1.5,0.0,['/projectnb/vkolagrp/datasets/BrainLat/data/M...
4,Alzheimer's disease,"{""Sex"": ""Male"", ""Age"": 72.0, ""Years of educati...",NaN,"{""MRI features extraction procedure"": ""A Gener...","{""Metadata"": {""Sex"": ""Male"", ""Age"": 72.0, ""Yea...",sub-PE00056,sub-PE00056,NaN,AD,1,72.0,15.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.0,3.0,2.0,2.0,3.0,2.0,3.0,2.0,4.0,11.0,10.0,NaN,1.0,0.0,0.0,1.5,0.0,['/projectnb/vkolagrp/datasets/BrainLat/data/M...


In [68]:
len(brainlat_merged)

660

In [69]:
def add_cogstate_brainlat(row):
    if row['Diagnosis'] in ["Alzheimer's disease", 'Behavioral variant frontotemporal dementia']:
        row['NACCUDSD'] = 4
    elif row['Diagnosis'] == 'Healthy':
        row['NACCUDSD'] = 1
    else:
        row['NACCUDSD'] = 3
        
    return row
        
brainlat_merged = brainlat_merged.apply(add_cogstate_brainlat, axis=1)

In [70]:
brainlat_dict = {
    "NACCUDSD": {
        "FORM ID": "D1",
        "Description": "Cognitive status at UDS visit",
        "Values": {
            "1": "Normal cognition",
            "3": "Parkinsons Disease",
            "4": "Dementia"
        }
    }
    
}

In [71]:
brainlat_merged['Diagnosis'].value_counts()

Diagnosis
Alzheimer's disease                           269
Healthy                                       242
Behavioral variant frontotemporal dementia    149
Name: count, dtype: int64

In [72]:
brainlat_merged['NACCUDSD'].value_counts()

NACCUDSD
4    418
1    242
Name: count, dtype: int64

In [73]:
len(brainlat_merged)

660

In [74]:
brainlat_merged.columns

Index(['Diagnosis', 'Metadata', 'EEG Report', 'MRI Report', 'patient_summary',
       'ID', 'MRI_ID', 'EEG_ID', 'diagnosis', 'sex', 'Age', 'years_education',
       'laterality', 'moca_total', 'moca_visuospatial', 'moca_recog',
       'moca_attention', 'moca_language', 'moca_abstraction', 'moca_memory',
       'moca_orientation', 'ifs_total_score', 'ifs_motor_series',
       'ifs_conflicting_instructions', 'ifs_motor_inhibition', 'ifs_digits',
       'ifs_months', 'ifs_visual_wm', 'ifs_proverb', 'ifs_verbal_inhibition',
       'mini_sea_fer', 'mini_sea_tom', 'emotion recog', 'T1', 'Rest', 'DWI',
       'MF', 'eeg', 'paths', 'NACCUDSD'],
      dtype='object')

In [75]:
def brainlat_stats(df, group_label="Overall"):
    """
    Collects statistics for NACC dataset and returns a dictionary.
    Rounds values to 2 decimal places where appropriate.
    """
    stats_dict = {"Group": group_label}
    
    # Create a copy to avoid modifying the original dataframe
    df = df.copy()
    
    # AGE
    col = df["Age"]
    stats_dict["AGE_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["AGE_Mean"] = round(col.mean(skipna=True), 2) if not col.isna().all() else None
    stats_dict["AGE_Std"] = round(col.std(skipna=True), 2) if not col.isna().all() else None
    stats_dict["AGE_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["AGE_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # SEX
    col = df["sex"]
    total = df.shape[0]
    count_female = (col == 0).sum()
    stats_dict["SEX_Female_Count"] = int(count_female)
    stats_dict["SEX_Female_Pct"] = round((count_female / total) * 100, 2) if total > 0 else 0
    stats_dict["SEX_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["SEX_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["SEX_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # EDUC
    col = df["years_education"]
    stats_dict["EDUC_NaN_pct"] = round(col.isna().mean() * 100, 2)
    stats_dict["EDUC_Mean"] = round(col.mean(skipna=True), 2) if not col.isna().all() else None
    stats_dict["EDUC_Std"] = round(col.std(skipna=True), 2) if not col.isna().all() else None
    stats_dict["EDUC_Min"] = round(col.min(), 2) if not col.isna().all() else None
    stats_dict["EDUC_Max"] = round(col.max(), 2) if not col.isna().all() else None
    
    # RACE
    # col = df["PTRACCAT"]
    # stats_dict["RACE_NaN_pct"] = round(col.isna().mean() * 100, 2)
    # stats_dict["RACE_Min"] = round(col.min(), 2) if not col.isna().all() else None
    # stats_dict["RACE_Max"] = round(col.max(), 2) if not col.isna().all() else None
    # # Get value counts for RACE
    # race_counts = col.value_counts(dropna=False).sort_index()
    # for val, count in race_counts.items():
    #     if pd.isna(val):
    #         stats_dict["RACE_NaN_Count"] = int(count)
    #     else:
    #         stats_dict[f"RACE_{adni_dict['PTRACCAT']['Values'][str(int(val))]}_Count"] = int(count)
    
    # CDR
    # df["CDR_TOT"] = df["CDR_TOT"].replace(-5.0, np.nan)
    # col = df['CDMEMORY']
    # stats_dict["CDR_NaN_pct"] = round(col.isna().mean() * 100, 2)
    # stats_dict["CDR_Mean"] = round(col.mean(skipna=True), 2) if not col.isna().all() else None
    # stats_dict["CDR_Std"] = round(col.std(skipna=True), 2) if not col.isna().all() else None
    # stats_dict["CDR_Min"] = round(col.min(), 2) if not col.isna().all() else None
    # stats_dict["CDR_Max"] = round(col.max(), 2) if not col.isna().all() else None

    return stats_dict

In [76]:
# Collect statistics for overall dataset
brainlat_stats_list = []
brainlat_stats_list.append(brainlat_stats(brainlat_merged, group_label="Overall"))

In [77]:
# Collect statistics for each NACCUDSD subgroup
for value in sorted(brainlat_merged['NACCUDSD'].dropna().unique()):
    subgroup_df = brainlat_merged[brainlat_merged['NACCUDSD'] == value].reset_index(drop=True)
    brainlat_stats_list.append(brainlat_stats(subgroup_df, group_label=f"{brainlat_dict['NACCUDSD']['Values'][str(int(value))]}"))

In [78]:
brainlat_merged[brainlat_merged['NACCUDSD'].isna()]

,Diagnosis,Metadata,EEG Report,MRI Report,patient_summary,ID,MRI_ID,EEG_ID,diagnosis,sex,Age,years_education,laterality,moca_total,moca_visuospatial,moca_recog,moca_attention,moca_language,moca_abstraction,moca_memory,moca_orientation,ifs_total_score,ifs_motor_series,ifs_conflicting_instructions,ifs_motor_inhibition,ifs_digits,ifs_months,ifs_visual_wm,ifs_proverb,ifs_verbal_inhibition,mini_sea_fer,mini_sea_tom,emotion recog,T1,Rest,DWI,MF,eeg,paths,NACCUDSD


In [79]:
# Create DataFrame from all collected statistics
brainlat_stats_list = pd.DataFrame(brainlat_stats_list)
brainlat_stats_list

,Group,AGE_NaN_pct,AGE_Mean,AGE_Std,AGE_Min,AGE_Max,SEX_Female_Count,SEX_Female_Pct,SEX_NaN_pct,SEX_Min,SEX_Max,EDUC_NaN_pct,EDUC_Mean,EDUC_Std,EDUC_Min,EDUC_Max
0,Overall,9.39,69.47,9.56,0.0,98.0,394,59.70,0.0,0,1,0.91,13.16,4.80,0.0,27.0
1,Normal cognition,17.36,67.88,8.88,39.0,92.0,159,65.70,0.0,0,1,0.41,14.72,4.31,3.0,27.0
2,Dementia,4.78,70.27,9.79,0.0,98.0,235,56.22,0.0,0,1,1.20,12.25,4.85,0.0,27.0


In [80]:
brainlat_stats_list.to_csv("../data/stats/brainlat.csv", index=False)

In [81]:
# Collect statistical test results
brainlat_test_results = {
    "Group": "Brainlat",
    "AGE_ANOVA_pvalue": anova_test(brainlat_merged, "Age", "NACCUDSD"),
    "SEX_ChiSquare_pvalue": chi_square_test(brainlat_merged, "sex", "NACCUDSD"),
    "EDUC_ANOVA_pvalue": anova_test(brainlat_merged, "years_education", "NACCUDSD"),
    # "RACE_ChiSquare_pvalue": chi_square_test(adni, "PTRACCAT", "NACCUDSD"),
    # "CDR_ChiSquare_pvalue": chi_square_test(adni, "CDMEMORY", "NACCUDSD")
}
# nacc_stats_list.append(test_results)
brainlat_test_results


{'Group': 'Brainlat',
 'AGE_ANOVA_pvalue': '3.803e-03',
 'SEX_ChiSquare_pvalue': '2.084e-02',
 'EDUC_ANOVA_pvalue': '1.288e-10'}

# Merge test results

In [75]:

# Collect all variables ending with "_test_results" into a dataframe
test_results_vars = [var for var in globals() if var.endswith('_test_results') and isinstance(globals()[var], dict)]
test_results_list = [globals()[var] for var in test_results_vars]
all_test_results_df = pd.DataFrame(test_results_list)
all_test_results_df

,Group,AGE_ANOVA_pvalue,SEX_ChiSquare_pvalue,EDUC_ANOVA_pvalue,RACE_ChiSquare_pvalue,CDR_ChiSquare_pvalue
0,NACC,3.467e-202,4.554e-166,4.004e-17,1.872e-130,0.000e+00
1,NIFD,2.634e-01,7.650e-03,6.880e-07,2.631e-04,2.894e-37
2,PPMI,1.615e-15,9.065e-08,7.121e-05,9.674e-01,NaN
3,ADNI,5.369e-05,2.650e-05,1.487e-07,9.932e-05,0.000e+00
4,Brainlat,1.335e-02,3.042e-06,1.220e-11,NaN,NaN


In [76]:
all_test_results_df.to_csv("../data/stats/stat_tests.csv", index=False)